# Advanced PySpark Optimization — Examples 31–40

Notebook 4 of the 5-notebook Advanced PySpark Optimization series.

## Examples

31. Broadcast join trade-offs and broadcast hints
32. Sort-merge join planning
33. Bucket-aware join setup
34. Repartitioning before a join
35. Join skew detection
36. Join skew mitigation with key salting
37. Catalyst optimization walkthrough
38. Predicate pushdown
39. Partition pruning vs. predicate pushdown
40. Built-in functions vs. Python UDFs

## Important lab note

The datasets are intentionally tiny so every example can be read and understood easily.

Therefore:

- join timings are not production benchmarks
- sort-merge join may not always be chosen unless configuration/statistics support it
- bucket-aware optimization may vary by Spark version and environment
- skew mitigation is demonstrated mechanically, not benchmarked
- predicate pushdown depends on the data source and file format

The focus is on **plans, data movement, optimizer visibility, and optimization reasoning**.

## Source mapping

This notebook combines the source material's advanced sections around:

- join strategy selection and trade-offs
- broadcast joins
- sort-merge joins
- bucketing and joins
- repartitioning for join preparation
- skew detection and mitigation
- Catalyst planning and optimization
- predicate pushdown
- partition pruning
- built-in functions versus Python UDFs

The examples continue the same progression used in the earlier notebooks: inspect the baseline, understand the execution mechanism, apply an optimization, and inspect the resulting plan.

In [ ]:
from pathlib import Path
import shutil

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType
)
from pyspark.sql.functions import udf

BASE_PATH = Path.cwd()
DATA_PATH = BASE_PATH / "data"

if not DATA_PATH.exists():
    candidate = Path("/mnt/data/pyspark_examples_31_40/data")
    if candidate.exists():
        DATA_PATH = candidate

assert DATA_PATH.exists(), "Could not find data/. Update BASE_PATH."

WORK_PATH = BASE_PATH / "work"
if str(BASE_PATH).startswith("/mnt/data/pyspark_examples_31_40"):
    WORK_PATH = BASE_PATH / "work"

if WORK_PATH.exists():
    shutil.rmtree(WORK_PATH)
WORK_PATH.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("Advanced-PySpark-Examples-31-40")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.warehouse.dir", str(WORK_PATH / "warehouse"))
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("segment", StringType(), True),
])

products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("brand", StringType(), True),
])

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("order_date", StringType(), True),
    StructField("region", StringType(), True),
])

customers_df = (
    spark.read.option("header", True)
    .schema(customers_schema)
    .csv(str(DATA_PATH / "customers.csv"))
)

products_df = (
    spark.read.option("header", True)
    .schema(products_schema)
    .csv(str(DATA_PATH / "products.csv"))
)

orders_df = (
    spark.read.option("header", True)
    .schema(orders_schema)
    .csv(str(DATA_PATH / "orders.csv"))
)

print("Spark version:", spark.version)
print("Orders:", orders_df.count())
print("Customers:", customers_df.count())
print("Products:", products_df.count())

# Example 31 — Broadcast Join Trade-Offs and Broadcast Hints

**Concepts:** explicit broadcast, small dimension tables, memory trade-offs.

Broadcasting avoids shuffling the large side of a join by sending a small table to executors.

The trade-off is that the broadcasted relation must be safely handled in executor memory.

This example compares:

- Spark's automatic join planning
- an explicit broadcast hint

Inspect the plans rather than relying on timing.

In [ ]:
auto_join_df = (
    orders_df
    .join(customers_df, "customer_id")
    .select("order_id", "customer_id", "customer_name", "segment", "amount")
)

explicit_broadcast_df = (
    orders_df
    .join(F.broadcast(customers_df), "customer_id")
    .select("order_id", "customer_id", "customer_name", "segment", "amount")
)

print("AUTOMATIC JOIN PLAN")
auto_join_df.explain("formatted")

print("\nEXPLICIT BROADCAST JOIN PLAN")
explicit_broadcast_df.explain("formatted")

assert auto_join_df.count() == explicit_broadcast_df.count()

print("\nTrade-off:")
print("- Broadcast reduces redistribution of the large side.")
print("- The broadcast relation consumes executor memory.")
print("- Do not broadcast a table just because broadcasting is available.")

# Example 32 — Sort-Merge Join Planning

**Concepts:** sort-merge joins, shuffle redistribution, join strategy configuration.

We temporarily disable automatic broadcast and enable the sort-merge join preference, then inspect the plan.

The exact physical strategy can still depend on Spark version and planning details. The important lesson is to inspect the resulting plan for:

- `SortMergeJoin`
- `Exchange`
- `Sort`

In [ ]:
original_threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
original_smj_preference = spark.conf.get("spark.sql.join.preferSortMergeJoin")

try:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    spark.conf.set("spark.sql.join.preferSortMergeJoin", "true")

    sort_merge_candidate = (
        orders_df
        .join(products_df, "product_id")
        .select("order_id", "product_id", "product", "category", "amount")
    )

    print("SORT-MERGE JOIN CANDIDATE PLAN")
    sort_merge_candidate.explain("formatted")

    sort_merge_candidate.orderBy("order_id").show()

finally:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", original_threshold)
    spark.conf.set("spark.sql.join.preferSortMergeJoin", original_smj_preference)

# Example 33 — Bucket-Aware Join Setup

**Concepts:** bucketing, compatible join keys, persisted physical layout.

We create two bucketed tables using the same bucket count and join key.

This demonstrates the setup required for bucket-aware execution. Whether Spark eliminates a shuffle depends on the runtime environment and optimizer configuration.

Inspect:

- table metadata
- join plan
- exchanges/shuffles, if any

In [ ]:
spark.sql("DROP TABLE IF EXISTS orders_bucketed_join")
spark.sql("DROP TABLE IF EXISTS products_bucketed_join")

(
    orders_df
    .write
    .format("parquet")
    .mode("overwrite")
    .bucketBy(4, "product_id")
    .sortBy("product_id")
    .saveAsTable("orders_bucketed_join")
)

(
    products_df
    .write
    .format("parquet")
    .mode("overwrite")
    .bucketBy(4, "product_id")
    .sortBy("product_id")
    .saveAsTable("products_bucketed_join")
)

bucketed_orders_df = spark.table("orders_bucketed_join")
bucketed_products_df = spark.table("products_bucketed_join")

bucketed_join_df = (
    bucketed_orders_df
    .join(bucketed_products_df, "product_id")
    .select("order_id", "product_id", "product", "category", "amount")
)

print("ORDERS BUCKETED TABLE METADATA")
spark.sql("DESCRIBE EXTENDED orders_bucketed_join").show(truncate=False)

print("\nPRODUCTS BUCKETED TABLE METADATA")
spark.sql("DESCRIBE EXTENDED products_bucketed_join").show(truncate=False)

print("\nBUCKET-AWARE JOIN PLAN")
bucketed_join_df.explain("formatted")

bucketed_join_df.orderBy("order_id").show()

# Example 34 — Repartitioning Before a Join

**Concepts:** runtime data distribution, join-key partitioning, shuffle preparation.

Repartitioning both inputs by the join key can be useful in some multi-step workloads, but it can also add unnecessary shuffles.

This example compares:

- a direct join
- inputs explicitly repartitioned by the join key

The lesson is not "always repartition before joining." The lesson is to inspect the full pipeline and avoid redundant exchanges.

In [ ]:
direct_join_df = (
    orders_df
    .join(products_df, "product_id")
)

repartitioned_orders = orders_df.repartition(4, "product_id")
repartitioned_products = products_df.repartition(4, "product_id")

prepared_join_df = (
    repartitioned_orders
    .join(repartitioned_products, "product_id")
)

print("DIRECT JOIN PLAN")
direct_join_df.explain("formatted")

print("\nREPARTITION-PREPARED JOIN PLAN")
prepared_join_df.explain("formatted")

assert direct_join_df.count() == prepared_join_df.count()
print("\nValidation passed.")

# Example 35 — Join Skew Detection

**Concepts:** skewed keys, partition imbalance, stragglers.

The source orders data already contains repeated activity for `C001`, but a 10-row dataset is too small to make skew operationally dramatic.

We inspect key frequency and partition distribution after repartitioning by the join key.

In [ ]:
print("JOIN-KEY FREQUENCY")
orders_df.groupBy("customer_id").count().orderBy(F.desc("count")).show()

repartitioned_by_customer = orders_df.repartition(4, "customer_id")

partition_distribution = (
    repartitioned_by_customer.rdd
    .mapPartitionsWithIndex(
        lambda partition_id, rows: [(partition_id, sum(1 for _ in rows))]
    )
    .collect()
)

print("\nROWS PER PARTITION")
for partition_id, row_count in sorted(partition_distribution):
    print(f"Partition {partition_id}: {row_count} rows")

print("\nSkew investigation checklist:")
print("- Look for keys with unusually high frequency.")
print("- Look for a small number of long-running tasks.")
print("- Compare partition sizes.")
print("- Inspect Spark UI task-duration and shuffle metrics.")

# Example 36 — Join Skew Mitigation with Key Salting

**Concepts:** key salting, distributing a hot key, controlled replication of the small side.

This example demonstrates the mechanics:

1. assign multiple salt values to the skewed/hot key on the large side
2. replicate only the matching hot-key dimension row across those salt values
3. join using `(customer_id, salt)`

For tiny data, this is an educational pattern rather than a performance benchmark.

Salting has trade-offs: it increases complexity and can increase the amount of data on the replicated side.

In [ ]:
HOT_KEY = "C001"
SALT_BUCKETS = 4

# Large/fact side: assign a deterministic salt for the hot key.
salted_orders_df = (
    orders_df
    .withColumn(
        "salt",
        F.when(
            F.col("customer_id") == HOT_KEY,
            F.pmod(F.hash("order_id"), F.lit(SALT_BUCKETS))
        ).otherwise(F.lit(0))
    )
)

# Small/dimension side: replicate only the hot key across all salt values.
hot_customer_df = (
    customers_df
    .filter(F.col("customer_id") == HOT_KEY)
    .withColumn("salt", F.explode(F.sequence(F.lit(0), F.lit(SALT_BUCKETS - 1))))
)

non_hot_customer_df = (
    customers_df
    .filter(F.col("customer_id") != HOT_KEY)
    .withColumn("salt", F.lit(0))
)

salted_customers_df = hot_customer_df.unionByName(non_hot_customer_df)

salted_join_df = (
    salted_orders_df
    .join(salted_customers_df, ["customer_id", "salt"])
)

print("SALT DISTRIBUTION FOR HOT KEY")
salted_orders_df.filter(F.col("customer_id") == HOT_KEY).groupBy("salt").count().orderBy("salt").show()

print("\nREPLICATED DIMENSION ROWS FOR HOT KEY")
salted_customers_df.filter(F.col("customer_id") == HOT_KEY).orderBy("salt").show()

print("\nSALTED JOIN PLAN")
salted_join_df.explain("formatted")

print("\nSALTED JOIN RESULT")
salted_join_df.orderBy("order_id").show()

assert salted_join_df.count() == orders_df.join(customers_df, "customer_id").count()
print("Row-count validation passed.")

# Example 37 — Catalyst Optimization Walkthrough

**Concepts:** parsed/logical plan, analyzed plan, optimized logical plan, physical plan.

`explain("extended")` makes Spark's planning pipeline visible.

The pipeline includes filters, projections, expressions, joins, and aggregation so that optimizer rewrites have something meaningful to inspect.

In [ ]:
catalyst_df = (
    orders_df
    .join(products_df, "product_id")
    .filter(
        (F.col("category") == "Electronics") &
        (F.col("amount") > 500)
    )
    .select(
        "region",
        "brand",
        "amount",
        (F.col("amount") * F.lit(0.9)).alias("discounted_amount")
    )
    .groupBy("region", "brand")
    .agg(F.sum("discounted_amount").alias("revenue_after_discount"))
)

print("EXTENDED PLAN")
catalyst_df.explain("extended")

print("\nRESULT")
catalyst_df.show()

# Example 38 — Predicate Pushdown

**Concepts:** pushing filters toward the data source, reducing data read.

CSV is not a good demonstration of predicate pushdown, so this example writes the data to Parquet and then reads it with a filter.

Inspect the file scan section of the plan for pushed filters.

Exact plan wording depends on Spark version, but Parquet-based scans typically expose pushed filter information in the scan metadata.

In [ ]:
parquet_orders_path = WORK_PATH / "orders_parquet"

orders_df.write.mode("overwrite").parquet(str(parquet_orders_path))

predicate_pushdown_df = (
    spark.read.parquet(str(parquet_orders_path))
    .filter(F.col("amount") > 500)
    .select("order_id", "customer_id", "amount")
)

print("PARQUET FILTER PLAN")
predicate_pushdown_df.explain("formatted")

print("\nRESULT")
predicate_pushdown_df.show()

# Example 39 — Partition Pruning vs. Predicate Pushdown

**Concepts:** physical partition elimination versus file-level predicate filtering.

These are related but different optimizations:

- **Partition pruning** avoids reading irrelevant partition directories.
- **Predicate pushdown** attempts to push a filter into the data source/file scan.

We create a Parquet dataset partitioned by `region` and compare:

1. filtering by partition column
2. filtering by a non-partition column

In [ ]:
partitioned_orders_path = WORK_PATH / "orders_by_region"

(
    orders_df
    .write
    .mode("overwrite")
    .partitionBy("region")
    .parquet(str(partitioned_orders_path))
)

region_pruning_df = (
    spark.read.parquet(str(partitioned_orders_path))
    .filter(F.col("region") == "East")
)

amount_pushdown_df = (
    spark.read.parquet(str(partitioned_orders_path))
    .filter(F.col("amount") > 500)
)

print("PARTITION-COLUMN FILTER — LOOK FOR PARTITION FILTERS")
region_pruning_df.explain("formatted")

print("\nNON-PARTITION FILTER — LOOK FOR PUSHED FILTERS")
amount_pushdown_df.explain("formatted")

print("\nPartition-pruned result:")
region_pruning_df.show()

print("\nPredicate-filtered result:")
amount_pushdown_df.show()

# Example 40 — Built-In Functions vs. Python UDFs

**Concepts:** optimizer visibility, Python UDF overhead, built-in expression optimization.

The same amount-band logic is implemented twice:

1. using Spark SQL/DataFrame built-in expressions
2. using a Python UDF

Built-in expressions are generally more visible to Spark's optimizer and avoid Python execution boundaries that UDFs introduce.

Inspect the plans and compare the operators.

In [ ]:
# Built-in expression version
built_in_df = (
    orders_df
    .withColumn(
        "amount_band",
        F.when(F.col("amount") >= 1000, F.lit("HIGH"))
         .when(F.col("amount") >= 500, F.lit("MEDIUM"))
         .otherwise(F.lit("LOW"))
    )
)

# Python UDF version
@udf(returnType=StringType())
def amount_band_udf(amount):
    if amount is None:
        return None
    if amount >= 1000:
        return "HIGH"
    if amount >= 500:
        return "MEDIUM"
    return "LOW"

python_udf_df = (
    orders_df
    .withColumn("amount_band", amount_band_udf(F.col("amount")))
)

print("BUILT-IN EXPRESSION PLAN")
built_in_df.explain("formatted")

print("\nPYTHON UDF PLAN")
python_udf_df.explain("formatted")

print("\nBUILT-IN RESULT")
built_in_df.orderBy("order_id").show()

print("\nPYTHON UDF RESULT")
python_udf_df.orderBy("order_id").show()

built_in_result = {
    row["order_id"]: row["amount_band"]
    for row in built_in_df.select("order_id", "amount_band").collect()
}

python_udf_result = {
    row["order_id"]: row["amount_band"]
    for row in python_udf_df.select("order_id", "amount_band").collect()
}

assert built_in_result == python_udf_result
print("Result validation passed.")

# Notebook 4 Summary

Examples 31–40 focused on **advanced joins and optimizer visibility**.

## Join optimization

- broadcast trade-offs
- sort-merge join planning
- bucket-aware join setup
- repartitioning by join keys
- skew detection
- key salting for skew mitigation

## Catalyst and data-source optimization

- parsed, analyzed, optimized, and physical plans
- predicate pushdown
- partition pruning
- built-in Spark expressions
- Python UDF execution boundaries

## Core optimization habit

For joins:

1. understand data sizes
2. inspect the join plan
3. identify redistribution
4. check key distribution
5. use the smallest appropriate optimization
6. validate results

For expressions:

1. prefer built-in Spark functions where possible
2. keep logic visible to Catalyst
3. inspect plans before and after introducing UDFs

## Final notebook preview

Examples 41–50 will cover the final advanced section:

- AQE off vs. AQE on
- adaptive shuffle partition coalescing
- AQE skew join handling
- dynamic partition pruning
- runtime filtering / Bloom-filter concepts
- end-to-end AQE optimization
- window-function optimization
- complex plan optimization
- production troubleshooting workflow
- end-to-end optimization case study